In [ ]:
!pip install -q timm kagglehub librosa

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import timm

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm

In [ ]:
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")

In [4]:
path = kagglehub.competition_download('itmo-acoustic-event-detectin-2026')
print("Path to competition files:", path)

competition_path = path

train_folder = os.path.join(competition_path, "audio_train", "train")
train_csv = os.path.join(competition_path, "train.csv")

df = pd.read_csv(train_csv, skiprows=1, names=["fname", "label"])
df["path"] = df["fname"].apply(lambda x: os.path.join(train_folder, x))

labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
df["label_id"] = df["label"].map(label2id)

Path to competition files: /root/.cache/kagglehub/competitions/itmo-acoustic-event-detectin-2026


In [6]:
class MelDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df
        self.augment = augment
        self.sr = 32000
        self.n_mels = 128
        self.max_len = self.sr * 5

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, _ = librosa.load(row["path"], sr=self.sr)

        # crop / pad
        if len(wav) < self.max_len:
            wav = np.pad(wav, (0, self.max_len - len(wav)))
        else:
            start = np.random.randint(0, len(wav) - self.max_len + 1)
            wav = wav[start:start + self.max_len]

        # mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=wav, sr=self.sr, n_mels=self.n_mels
        )
        mel = librosa.power_to_db(mel).astype(np.float32)

        # normalization
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        # 3 канала
        mel = np.stack([mel, mel, mel], axis=0)

        return torch.tensor(mel), torch.tensor(row["label_id"])

In [7]:
train_df, val_df = train_test_split(
    df, test_size=0.1, stratify=df["label_id"], random_state=42
)

train_ds = MelDataset(train_df, augment=True)
val_ds = MelDataset(val_df, augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=len(label2id)
).to(device)

In [11]:
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

EPOCHS = 15
best_f1 = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}")

    # TRAIN
    model.train()
    train_preds, train_targets = [], []

    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)

        x, y_a, y_b, lam = mixup(x, y)

        optimizer.zero_grad()
        outputs = model(x)

        loss = mixup_loss(criterion, outputs, y_a, y_b, lam)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)
        train_preds.extend(preds.cpu().numpy())
        train_targets.extend(y.cpu().numpy())

    train_f1 = f1_score(train_targets, train_preds, average="weighted")

    # VALIDATION
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for x, y in tqdm(val_loader):
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            preds = outputs.argmax(1)

            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(y.cpu().numpy())

    val_f1 = f1_score(val_targets, val_preds, average="weighted")

    print(f"Train F1: {train_f1:.4f}")
    print(f"Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_efficientnet.pt")
        print("Saved best model")

print("Best F1:", best_f1)


Epoch 1


100%|██████████| 18/18 [00:46<00:00,  2.56s/it]


Train F1: 0.2344
Val F1: 0.5771
Saved best model

Epoch 2


100%|██████████| 18/18 [00:48<00:00,  2.69s/it]


Train F1: 0.3669
Val F1: 0.6829
Saved best model

Epoch 3


...

Train F1: 0.4949
Val F1: 0.7784

Epoch 14


100%|██████████| 18/18 [00:54<00:00,  3.02s/it]


Train F1: 0.4814
Val F1: 0.7781

Epoch 15


100%|██████████| 18/18 [00:53<00:00,  2.97s/it]


Train F1: 0.4926
Val F1: 0.7914
Saved best model
Best F1: 0.7913968591512145


In [29]:
torch.save(model.state_dict(), "best_efficientnet.pt")

## Тестирование

In [ ]:
test_folder = os.path.join(competition_path, "audio_test", "test")
test_files = sorted(os.listdir(test_folder))

print("Test files:", len(test_files))

Test files: 3790


In [49]:
TARGET_SR = 32000
MAX_LEN = TARGET_SR * 5

def load_audio(path):
    wav, sr = librosa.load(path, sr=TARGET_SR)

    if len(wav) < MAX_LEN:
        wav = np.pad(wav, (0, MAX_LEN - len(wav)))
    else:
        start = np.random.randint(0, len(wav) - MAX_LEN + 1)
        wav = wav[start:start + MAX_LEN]

    mel = librosa.feature.melspectrogram(y=wav, sr=TARGET_SR, n_mels=128)
    mel = librosa.power_to_db(mel)

    mel = (mel - mel.mean()) / (mel.std() + 1e-6)

    return torch.tensor(np.stack([mel, mel, mel], axis=0), dtype=torch.float32)

In [50]:
model.eval()

preds = []
filenames = []

In [51]:
with torch.no_grad():
    for fname in tqdm(test_files):
        path = os.path.join(test_folder, fname)

        try:
            wav = load_audio(path)
            x = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).to(device)

            outputs = model(x)
            pred = outputs.argmax(dim=1).cpu().item()

            preds.append(pred)
            filenames.append(fname)

        except Exception as e:
            print("ERROR FILE:", fname)

  0%|          | 0/3790 [00:00<?, ?it/s]/tmp/ipykernel_13225/2408644597.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).to(device)
100%|██████████| 3790/3790 [05:58<00:00, 10.57it/s]


In [52]:
df = pd.read_csv(train_csv, skiprows=1, names=["fname", "label"])

labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

In [53]:
pred_labels = [id2label[p] for p in preds]

submission = pd.DataFrame({
    "fname": filenames,
    "label": pred_labels
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

                      fname                  label
0  001e64cdd9984e165d34.wav                   Oboe
1  001e949afb7a313b4cfe.wav       Violin_or_fiddle
2  005e20afb8ba6b054afb.wav                  Flute
3  007c5f37ab13c09bed62.wav              Harmonica
4  008736a59a001197a5d2.wav  Burping_or_eructation
